<a href="https://colab.research.google.com/github/sbooeshaghi/llmarkers/blob/main/analysis/extraction/method_cmp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install --quiet tiktoken unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.5/235.5 kB 11.3 MB/s eta 0:00:00


# Custom n-gram alignment

In [44]:
from collections import defaultdict

# from itertools import product
import unicodedata
from unidecode import unidecode

import tiktoken
import re


def norm_text(text):
    """
    Normalize text to ensure UTF-8 compatibility for NLP processing.

    This function:
    1. Normalizes Unicode to the NFC form
    2. Replaces problematic characters with ASCII equivalents
    3. Handles common special characters that cause issues

    Args:
        text (str): Input text to normalize

    Returns:
        str: Normalized text safe for UTF-8 processing
    """
    if not isinstance(text, str):
        try:
            text = str(text)
        except:
            return ""

    # Step 1: Unicode normalization to NFC form (composed form)
    # This combines characters and diacritics when possible
    text = unicodedata.normalize("NFC", text)
    text = unidecode(text)

    # Step 2: Map specific problematic characters to ASCII equivalents
    char_map = {
        "–": "-",  # en dash
        "—": "--",  # em dash
        "‘": "'",  # left single quote
        "’": "'",  # right single quote
        "“": '"',  # left double quote
        "”": '"',  # right double quote
        "…": "...",  # ellipsis
        "•": "*",  # bullet
        "·": ".",  # middle dot
        "×": "x",  # multiplication sign
        "÷": "/",  # division sign
        "≤": "<=",  # less than or equal
        "≥": ">=",  # greater than or equal
        "≠": "!=",  # not equal
        "≈": "~",  # approximately equal
        "∞": "inf",  # infinity
        "∂": "d",  # partial differential
        "∫": "integral",  # integral
        "∑": "sum",  # sum
        "∏": "product",  # product
        "√": "sqrt",  # square root
        "∝": "prop to",  # proportional to
        "∠": "angle",  # angle
        "△": "triangle",  # triangle
        "□": "square",  # square
        "∈": "in",  # element of
        "∉": "not in",  # not an element of
        "⊂": "subset",  # subset
        "⊃": "superset",  # superset
        "∪": "union",  # union
        "∩": "intersect",  # intersection
        "⊆": "subseteq",  # subset or equal
        "⊇": "superseteq",  # superset or equal
    }

    for char, replacement in char_map.items():
        text = text.replace(char, replacement)

    # Step 3: Remove any remaining non-ASCII characters (optional)
    # Uncomment if you want to remove ALL non-ASCII characters
    # text = re.sub(r'[^\x00-\x7F]+', '', text)

    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)  # .strip()

    return text


def tokenize_with_offsets(text, encoding="cl100k_base"):
    """Tokenizes text and returns tokens with their character start positions."""
    enc = tiktoken.get_encoding(encoding)
    tokens = []

    etks = enc.encode(text)
    dtks, ptks = enc.decode_with_offsets(etks)

    assert len(etks) == len(ptks)

    for t, p in zip(etks, ptks):
        token = enc.decode_single_token_bytes(t).decode("utf-8")
        tobj = {
            "token": token,
            "enc_token": t,
            "start_idx": p,
            "end_idx": p + len(token),
        }
        tokens.append(tobj)
    return tokens


def tokenize(text):
    nt = norm_text(text)
    tks = tokenize_with_offsets(nt)
    t2w = defaultdict(list)
    for tk in tks:
        t2w[tk["enc_token"]].append(tk)
    return (tks, t2w)


def build_index(text, k=1):
    tokens, t2w = tokenize(text)
    ngram_to_id = {}  # Maps n-grams to unique IDs
    id_to_ngram = {}  # Map unique IDs to n-grams
    ngram_id_to_pos = defaultdict(list)  # Positions of each n-gram id
    ngrams = []  # actual list of ngrams (and corresponding tokens)
    counter = 0

    enc_tokens = [i["enc_token"] for i in tokens]

    for i in range(len(enc_tokens) - k + 1):
        ngram = tuple(enc_tokens[i : i + k])  # encoded tuple of tokens

        # get the ngram id or add it
        if ngram not in ngram_to_id:
            ngram_to_id[ngram] = counter
            id_to_ngram[counter] = ngram
            counter += 1
        ngram_id = ngram_to_id[ngram]

        ngram_id_to_pos[ngram_id].append(i)

        ngrams.append({"ngram_id": ngram_id, "tks": tokens[i : i + k]})

    return (ngram_to_id, id_to_ngram, ngram_id_to_pos, ngrams)


def align_target(target, ngram_to_id, ngram_id_to_pos, k=1):
    tokens, t2w = tokenize(target)

    enc_tokens = [i["enc_token"] for i in tokens]

    aln = []

    prev_id = None
    for i in range(len(enc_tokens) - k + 1):
        target_ngram = tuple(enc_tokens[i : i + k])  # build the ngram from the target

        ngram_id = ngram_to_id.get(
            tuple(target_ngram), None
        )  # align the target ngram to the ngrams built from source

        if ngram_id is not None:
            if prev_id is not None and prev_id == ngram_id:
                continue  # Skip duplicate adjacent n-grams
            aln.append(
                {
                    target_ngram: [
                        {"ngram_id": ngram_id, "pos": j}
                        for j in ngram_id_to_pos[ngram_id]
                    ]
                }
            )
            prev_id = ngram_id

    return aln


# def group_ngrams(aln):
#     # group ngram combinations (ensure positions are ordered)
#     position_lists = [list(entry.values())[0] for entry in aln]

#     all_combinations = list(product(*position_lists))
#     grp = []
#     idx = 0
#     for combo in all_combinations:
#         positions = [item["pos"] for item in combo]
#         if positions == sorted(positions):  # Ensure positions are increasing
#             grp.append(list(combo))
#             idx += 1

#     return grp

# grouping ngrams with dfs pruning
def group_ngrams(aln):
    position_lists = [list(entry.values())[0] for entry in aln]

    def dfs(idx, path):
        if idx == len(position_lists):
            yield path
            return
        last_pos = path[-1]["pos"] if path else -1
        for candidate in position_lists[idx]:
            if candidate["pos"] > last_pos:
                yield from dfs(idx + 1, path + [candidate])

    grp = list(dfs(0, []))
    return grp


def group_tokens(grp, ngrams):
    # group tokens for each combination of ngrams
    tk_aln = []
    for aln_ngrams in grp:
        pos = aln_ngrams[0]["pos"]
        nid = aln_ngrams[0]["ngram_id"]

        tks = []
        tks += ngrams[pos]["tks"]

        for ng in aln_ngrams[1:]:
            pos = ng["pos"]
            nid = ng["ngram_id"]
            tks += [ngrams[pos]["tks"][-1]]
        tk_aln.append(tks)
    return tk_aln


def build_graph(ngrams, id_to_ngram):
    graph = defaultdict(set)
    for ng in ngrams:
        ngram_id = ng["ngram_id"]
        ngram = id_to_ngram[ngram_id]
        for existing_ngram_id, existing_ngram in id_to_ngram.items():
            if existing_ngram_id == ngram_id:
                continue  # Skip self

            # Check left adjacency (ngram[:-1] matches existing_ngram[1:])
            if ngram[:-1] == existing_ngram[1:]:
                graph[existing_ngram_id].add(ngram_id)

            # Check right adjacency (ngram[1:] matches existing_ngram[:-1])
            if ngram[1:] == existing_ngram[:-1]:
                graph[ngram_id].add(existing_ngram_id)
    return graph


def align_ng(source, target, k=1, verbose=False):
    if verbose:
        print("source", source)
        print("target", target)
        print("-" * 80)
    ngram_to_id, id_to_ngram, ngram_id_to_pos, ngrams = build_index(source, k)
    aln = align_target(target, ngram_to_id, ngram_id_to_pos, k)

    if len(aln) > 0:
        grp = group_ngrams(aln)
        tks = group_tokens(grp, ngrams)
        n_aln = len(tks)
        return tks
    return []


def reconstruct_target_by_token(source, pos):
    # reconstruct the target by joining the tokens
    return "".join([i["token"] for i in pos])


def reconstruct_target_by_idx(source, pos):
    # reconstruct the target by joining the tokens
    return "".join([source[i["start_idx"] : i["end_idx"]] for i in pos])


In [45]:
source = """
Upper and lower bounds are usually stated using the big O notation, which hides constant factors and smaller terms. This makes the bounds independent of the specific details of the computational model used. For instance, if T(n) = 7n2 + 15n + 40, in big O notation one would write T(n) = O(n2).
"""
target = " T(n) = O(n2)."
src_tokens, src_t2w = tokenize(source)

k = 1
alns = align_ng(source, target, k)

for idx, pos in enumerate(alns):
  recon = reconstruct_target_by_idx(source, pos)
  print(f"aln {idx} match ({target == recon}): '{target}' == '{recon}'")

aln 0 match (True): ' T(n) = O(n2).' == ' T(n) = O(n2).'
aln 1 match (True): ' T(n) = O(n2).' == ' T(n) = O(n2).'
aln 2 match (True): ' T(n) = O(n2).' == ' T(n) = O(n2).'
aln 3 match (True): ' T(n) = O(n2).' == ' T(n) = O(n2).'
aln 4 match (True): ' T(n) = O(n2).' == ' T(n) = O(n2).'
aln 5 match (True): ' T(n) = O(n2).' == ' T(n) = O(n2).'
aln 6 match (True): ' T(n) = O(n2).' == ' T(n) = O(n2).'


In [46]:
import difflib
import tiktoken

def align_difflib(source_text, target_text):
    source_tokens = tokenize_with_offsets(source_text)
    target_tokens = tokenize_with_offsets(target_text)

    source_enc_tokens = [t['enc_token'] for t in source_tokens]
    target_enc_tokens = [t['enc_token'] for t in target_tokens]

    matcher = difflib.SequenceMatcher(None, source_enc_tokens, target_enc_tokens)

    alignments = []
    for block in matcher.get_matching_blocks():
        if block.size > 0:
            alignment = source_tokens[block.a : block.a + block.size]
            alignments.append(alignment)
    # flatten the list of lists
    alignments = [item for sublist in alignments for item in sublist]

    return [alignments]

# Example usage:
source = "The quick brown fox jumps over the lazy dog."
target = "A quick fox jumps over a lazy dog."

alns = align_difflib(source, target)
alns

[[{'token': ' quick', 'enc_token': 4062, 'start_idx': 3, 'end_idx': 9},
  {'token': ' fox', 'enc_token': 39935, 'start_idx': 15, 'end_idx': 19},
  {'token': ' jumps', 'enc_token': 35308, 'start_idx': 19, 'end_idx': 25},
  {'token': ' over', 'enc_token': 927, 'start_idx': 25, 'end_idx': 30},
  {'token': ' lazy', 'enc_token': 16053, 'start_idx': 34, 'end_idx': 39},
  {'token': ' dog', 'enc_token': 5679, 'start_idx': 39, 'end_idx': 43},
  {'token': '.', 'enc_token': 13, 'start_idx': 43, 'end_idx': 44}]]

In [47]:
import numpy as np
import tiktoken

def align_lcs(source_text, target_text):
    source_tokens = tokenize_with_offsets(source_text)
    target_tokens = tokenize_with_offsets(target_text)

    source_enc_tokens = [t['enc_token'] for t in source_tokens]
    target_enc_tokens = [t['enc_token'] for t in target_tokens]

    m, n = len(source_enc_tokens), len(target_enc_tokens)
    dp = np.zeros((m+1, n+1), dtype=int)

    # DP computation
    for i in range(m):
        for j in range(n):
            if source_enc_tokens[i] == target_enc_tokens[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])

    # Backtracking to get LCS alignment
    i, j = m, n
    alignment = []
    while i > 0 and j > 0:
        if source_enc_tokens[i-1] == target_enc_tokens[j-1]:
            alignment.append(source_tokens[i-1])
            i -= 1
            j -= 1
        elif dp[i-1][j] >= dp[i][j-1]:
            i -= 1
        else:
            j -= 1

    alignment.reverse()
    return [alignment]  # wrapped in a list to match your desired structure

# Example usage:
source = "The quick brown fox jumps over the lazy dog."
target = "A quick fox jumps over a lazy dog."

alns = align_lcs(source, target)
alns

[[{'token': ' quick', 'enc_token': 4062, 'start_idx': 3, 'end_idx': 9},
  {'token': ' fox', 'enc_token': 39935, 'start_idx': 15, 'end_idx': 19},
  {'token': ' jumps', 'enc_token': 35308, 'start_idx': 19, 'end_idx': 25},
  {'token': ' over', 'enc_token': 927, 'start_idx': 25, 'end_idx': 30},
  {'token': ' lazy', 'enc_token': 16053, 'start_idx': 34, 'end_idx': 39},
  {'token': ' dog', 'enc_token': 5679, 'start_idx': 39, 'end_idx': 43},
  {'token': '.', 'enc_token': 13, 'start_idx': 43, 'end_idx': 44}]]

In [48]:
alns = align_ng(source, target)
alns

[[{'token': ' quick', 'enc_token': 4062, 'start_idx': 3, 'end_idx': 9},
  {'token': ' fox', 'enc_token': 39935, 'start_idx': 15, 'end_idx': 19},
  {'token': ' jumps', 'enc_token': 35308, 'start_idx': 19, 'end_idx': 25},
  {'token': ' over', 'enc_token': 927, 'start_idx': 25, 'end_idx': 30},
  {'token': ' lazy', 'enc_token': 16053, 'start_idx': 34, 'end_idx': 39},
  {'token': ' dog', 'enc_token': 5679, 'start_idx': 39, 'end_idx': 43},
  {'token': '.', 'enc_token': 13, 'start_idx': 43, 'end_idx': 44}]]

In [53]:
source = "hello my name is sina no my name is frank"
target = " name is"

In [54]:
align_difflib(source, target)

[[{'token': ' name', 'enc_token': 836, 'start_idx': 8, 'end_idx': 13},
  {'token': ' is', 'enc_token': 374, 'start_idx': 13, 'end_idx': 16}]]

In [55]:
align_lcs(source, target)

[[{'token': ' name', 'enc_token': 836, 'start_idx': 27, 'end_idx': 32},
  {'token': ' is', 'enc_token': 374, 'start_idx': 32, 'end_idx': 35}]]

In [56]:
align_ng(source, target)

[[{'token': ' name', 'enc_token': 836, 'start_idx': 8, 'end_idx': 13},
  {'token': ' is', 'enc_token': 374, 'start_idx': 13, 'end_idx': 16}],
 [{'token': ' name', 'enc_token': 836, 'start_idx': 8, 'end_idx': 13},
  {'token': ' is', 'enc_token': 374, 'start_idx': 32, 'end_idx': 35}],
 [{'token': ' name', 'enc_token': 836, 'start_idx': 27, 'end_idx': 32},
  {'token': ' is', 'enc_token': 374, 'start_idx': 32, 'end_idx': 35}]]